# 01 · Pipeline check with pretrained SmolVLA models (Step A3)

| model | expected | what it proves |
|---|---|---|
| `smolvla_base` | ~0% success, arm moves | plumbing works end to end |
| `smolvla_libero` (`HuggingFaceVLA/smolvla_libero`) | high success | our image flip, state, keys, normalization and gripper sign are right |

Run `00_colab_setup.ipynb` once first (it downloads the demos). A100 recommended; a T4 works but is slower.
Weights are downloaded to `MyDrive/groundedvla/models/` the first time, then copied to local disk each session.

In [1]:
# ---- edit once ----
REPO_URL = "https://github.com/kaustubhadhe1206/GroundedVLA.git"   # public repo, no token needed
BRANCH = "main"

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import os, subprocess
REPO = "/content/GroundedVLA"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
!git log -1 --format="commit %h  %s"

commit 8ca92d4  01_pipeline_check: smolvla_libero on T1-T7, skip done tasks, dedup summary per run


In [5]:
!apt-get -qq install -y libegl1 libgl1 > /dev/null
%cd /content/GroundedVLA
!git pull --ff-only && git log -1 --format="commit %h  %s"
!bash scripts/colab_install.sh requirements.txt

/content
Already up to date.
commit 8ca92d4  01_pipeline_check: smolvla_libero on T1-T7, skip done tasks, dedup summary per run
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuda-python 13.0.3 requires cuda-bindings~=13.0.3, but you have cuda-bindings 12.9.4 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,

In [6]:
from pathlib import Path
from groundedvla.utils.colab import setup_session, stage_to_local
paths = setup_session()          # env vars (EGL, caches on local disk), Drive folders, LIBERO config

profile=colab root=/content/drive/MyDrive/groundedvla commit=8ca92d4


In [7]:
# Fast local copies of weights already on Drive (first run: run_sim downloads them to Drive)
for folder in ["smolvla_base", "smolvla_libero"]:
    if (Path(paths["models"]) / folder / "config.json").exists():
        stage_to_local(f"models/{folder}")

staging /content/drive/MyDrive/groundedvla/models/smolvla_base -> /content/local/models/smolvla_base
staging /content/drive/MyDrive/groundedvla/models/smolvla_libero -> /content/local/models/smolvla_libero


## Rollouts (10 per task, videos saved to Drive)

`smolvla_base` only on T1 (plumbing, ~0%). `smolvla_libero` on all 7 tasks: this is the reference for our V1 baseline. Tasks that already have 10 logged episodes on Drive are skipped.

In [8]:
!pip install -q num2words==0.5.14

In [9]:
import json
EPISODES = 10
RUNS = [("smolvla_base", ["T1"]),                                      # plumbing only (~0%)
        ("smolvla_libero", ["T1", "T2", "T3", "T4", "T5", "T6", "T7"])]
SKIP_DONE = True     # skip model/task pairs that already have EPISODES results on Drive

def n_done(model, task):
    log = Path(paths["outputs"], "run_sim", model, f"{task}.jsonl")
    if not log.exists():
        return 0
    return len({json.loads(l)["episode_idx"] for l in log.read_text().splitlines() if l.strip()})

for model, tasks in RUNS:
    for task in tasks:
        if SKIP_DONE and n_done(model, task) >= EPISODES:
            print(f"skip {model} {task}: {n_done(model, task)} episodes already logged")
            continue
        !python scripts/run_sim.py {model} --task {task} --episodes {EPISODES} --viewer none --save-video

skip smolvla_base T1: 10 episodes already logged
skip smolvla_libero T1: 10 episodes already logged
skip smolvla_libero T2: 10 episodes already logged
skip smolvla_libero T3: 10 episodes already logged
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.13/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
[run_sim] model=smolvla_libero task=T4 (libero_goal) episodes=10 mode=naive N=1 max_steps=300 viewer=none profile=colab
I0000 00:00:1794709384.993844    7034 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...
config.json: 3.77kB [00:00, 7.07MB/s]


Optional: camera robustness preview (yaw ±20°, as in the light eval of T5–T7), and execution modes.

In [ ]:
# for task in ["T5", "T6", "T7"]:
#     for yaw in [20, -20]:
#         !python scripts/run_sim.py smolvla_libero --task {task} --episodes 10 --viewer none --camera-yaw {yaw} --save-video
# !python scripts/run_sim.py smolvla_libero --task T1 --episodes 10 --viewer none --horizon 1
# !python scripts/run_sim.py smolvla_libero --task T1 --episodes 10 --viewer none --mode temporal_ensemble

In [10]:
import json
from pathlib import Path
from IPython.display import Video, display, Markdown

MODEL = "smolvla_libero"
TASKS = ["T1", "T2", "T3", "T4", "T5", "T6", "T7"]   # or e.g. ["T5_yaw20"]
MAX_PER_TASK = 10

root = Path(paths["outputs"], "run_sim", MODEL)
for task in TASKS:
    # result of each episode from the log (a later run of the same episode wins)
    result = {}
    log = root / f"{task}.jsonl"
    if log.exists():
        for line in log.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                result[r["episode_idx"]] = r
    videos = sorted(root.glob(f"videos/{task}_ep*.mp4"))[:MAX_PER_TASK]
    display(Markdown(f"### {task}: {len(videos)} videos"))
    for v in videos:
        ep = int(v.stem.split("_ep")[1])
        r = result.get(ep, {})
        label = "SUCCESS" if r.get("success") else f"fail ({r.get('failure')})" if r else "?"
        print(f"{v.name}  {label}  steps={r.get('steps')}")
        display(Video(str(v), embed=True, width=512))


### T1: 10 videos

T1_ep000.mp4  SUCCESS  steps=83


T1_ep001.mp4  fail (failed_grasp)  steps=220


T1_ep002.mp4  fail (failed_grasp)  steps=220


T1_ep003.mp4  SUCCESS  steps=103


T1_ep004.mp4  fail (failed_grasp)  steps=220


... (67 outputs left out)

T7_ep006.mp4  SUCCESS  steps=110


T7_ep007.mp4  SUCCESS  steps=102


T7_ep008.mp4  SUCCESS  steps=96


T7_ep009.mp4  SUCCESS  steps=104


In [11]:
import json
import pandas as pd
rows = []
for f in Path(paths["outputs"], "run_sim").glob("*/*.jsonl"):
    for l in f.read_text().splitlines():
        if l.strip():
            rows.append({**json.loads(l), "model": f.parent.name, "run": f.stem})   # run = T3, T3_yaw20, ...
df = pd.DataFrame(rows).drop_duplicates(["model", "run", "episode_idx"], keep="last")   # a rerun replaces
df.groupby(["model", "run"]).agg(episodes=("success", "size"), success=("success", "mean"),
                                 steps=("steps", "mean"), policy_call_ms=("latency_ms_median", "median"))

episodes  success  steps  policy_call_ms
model          run                                          
replay         T1          5      1.0   79.8        0.016687
               T2          5      1.0  111.4        0.016284
               T3          5      1.0   86.8        0.016465
smolvla_base   T1         10      0.0  220.0      652.535217
               T2         10      0.0  220.0      652.168812
               T3         10      0.0  300.0      662.125645
smolvla_libero T1         10      0.4  164.8      888.962142
               T2         10      0.9  119.4      891.152574
               T3         10      0.9  108.6      887.716519
               T4         10      1.0   76.7      859.896909
               T5         10      1.0  131.5      874.667828
               T6         10      0.3  265.2      865.019634
               T7         10      0.9  117.6      859.501522